# AgentCore Gateway Demo — Centralized MCP with Auth Propagation

This notebook demonstrates **centralized MCP server management** with **Okta OIDC auth propagation** using AWS AgentCore Gateway.

## The "whoa" moments

1. **One Gateway endpoint** manages both Lambda functions AND MCP servers
2. **User identity flows through** — each user's JWT is validated by the Gateway
3. **Different users, different groups** — Alice (analyst) and Bob (finance-admin) authenticate separately
4. **Agent-level access control** — the agent checks user groups from the JWT to enforce policy

## Prerequisites

Run `01_setup.ipynb` first to create the infrastructure.

## Cell 1: Load Config + Pre-Authenticate Both Users

We authenticate both Alice and Bob upfront via Okta Resource Owner Password flow, then create separate Strands agents for each — one connected to the Gateway with Alice's token, one with Bob's.

The Gateway validates each JWT against Okta's JWKS endpoint and checks:
- **Audience** matches `api://default`
- **Client ID** matches the configured `allowedClients`
- **Signature** is valid per Okta's signing keys

In [ ]:
import os
import json
import jwt
import httpx
import requests
from dotenv import load_dotenv
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

# --- Load Gateway config from setup notebook ---
with open("gateway_config.json") as f:
    config = json.load(f)

GATEWAY_URL = config["gateway_url"]
OKTA_DOMAIN = config["okta_domain"]
OKTA_CLIENT_ID = config["okta_client_id"]
OKTA_ISSUER = config["okta_issuer"]
OKTA_CLIENT_SECRET = os.environ["OKTA_CLIENT_SECRET"]

# --- Okta credentials for test users ---
ALICE_USERNAME = os.environ["ALICE_USERNAME"]
ALICE_PASSWORD = os.environ["ALICE_PASSWORD"]
BOB_USERNAME = os.environ["BOB_USERNAME"]
BOB_PASSWORD = os.environ["BOB_PASSWORD"]

# --- Model to use for Strands agents ---
MODEL_ID = "global.anthropic.claude-sonnet-4-20250514-v1:0"


def get_okta_token(username: str, password: str) -> dict:
    """Authenticate a user via Okta Resource Owner Password flow and return the JWT."""
    token_url = f"{OKTA_ISSUER}/v1/token"
    response = requests.post(
        token_url,
        data={
            "grant_type": "password",
            "username": username,
            "password": password,
            "scope": "openid profile groups",
            "client_id": OKTA_CLIENT_ID,
            "client_secret": OKTA_CLIENT_SECRET,
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
    )
    response.raise_for_status()
    token_data = response.json()
    access_token = token_data["access_token"]

    # Decode JWT (without verification, just to display claims)
    claims = jwt.decode(access_token, options={"verify_signature": False})
    return {"access_token": access_token, "claims": claims}


# --- Pre-authenticate both users ---
print("Authenticating Alice (analyst group)...")
alice_auth = get_okta_token(ALICE_USERNAME, ALICE_PASSWORD)
print(f"  User: {alice_auth['claims'].get('sub', 'unknown')}")
print(f"  Groups: {alice_auth['claims'].get('groups', [])}")

print("\nAuthenticating Bob (finance-admins group)...")
bob_auth = get_okta_token(BOB_USERNAME, BOB_PASSWORD)
print(f"  User: {bob_auth['claims'].get('sub', 'unknown')}")
print(f"  Groups: {bob_auth['claims'].get('groups', [])}")

# --- Create MCPClients connected to Gateway with each user's token ---
alice_http = httpx.AsyncClient(headers={"Authorization": f"Bearer {alice_auth['access_token']}"})
bob_http = httpx.AsyncClient(headers={"Authorization": f"Bearer {bob_auth['access_token']}"})

alice_client = MCPClient(
    lambda: streamable_http_client(GATEWAY_URL, http_client=alice_http),
    startup_timeout=60,
)

bob_client = MCPClient(
    lambda: streamable_http_client(GATEWAY_URL, http_client=bob_http),
    startup_timeout=60,
)

# --- Create Strands Agents with group-aware system prompts ---
alice_client.__enter__()
bob_client.__enter__()

alice_groups = alice_auth["claims"].get("groups", [])
bob_groups = bob_auth["claims"].get("groups", [])

ALICE_SYSTEM_PROMPT = f"""You are a helpful assistant for user Alice.
Alice's group memberships: {alice_groups}

ACCESS POLICY:
- WeatherTools (get_weather): Available to ALL authenticated users
- FinanceTools (get_revenue_data): RESTRICTED to users in the 'finance-admins' group ONLY

Before calling any FinanceTools function, check the user's groups.
If the user is NOT in 'finance-admins', REFUSE the request and explain they don't have permission.
If a tool call fails with an access denied error, explain clearly that the user does not have permission."""

BOB_SYSTEM_PROMPT = f"""You are a helpful assistant for user Bob.
Bob's group memberships: {bob_groups}

ACCESS POLICY:
- WeatherTools (get_weather): Available to ALL authenticated users
- FinanceTools (get_revenue_data): RESTRICTED to users in the 'finance-admins' group ONLY

Before calling any FinanceTools function, check the user's groups.
If the user is NOT in 'finance-admins', REFUSE the request and explain they don't have permission.
Present data clearly when tools return results."""

alice_agent = Agent(
    model=MODEL_ID,
    tools=alice_client.list_tools_sync(),
    system_prompt=ALICE_SYSTEM_PROMPT,
)

bob_agent = Agent(
    model=MODEL_ID,
    tools=bob_client.list_tools_sync(),
    system_prompt=BOB_SYSTEM_PROMPT,
)

print(f"\n--- Gateway Authentication ---")
print(f"  Gateway URL: {GATEWAY_URL}")
print(f"  Alice authenticated: groups={alice_groups}")
print(f"  Bob authenticated:   groups={bob_groups}")
print(f"\n  Both agents ready!")
print(f"  Alice's agent: {len(alice_client.list_tools_sync())} tools available")
print(f"  Bob's agent:   {len(bob_client.list_tools_sync())} tools available")

## Cell 2: Show Available Tools

Both users see the same tools listed — the Gateway advertises all targets. Access control happens at **invocation time**, not at discovery time.

In [ ]:
print("Tools available through AgentCore Gateway:\n")

tools = alice_client.list_tools_sync()
for tool in tools:
    name = tool.tool_name if hasattr(tool, 'tool_name') else str(tool)
    print(f"  - {name}")

print(f"\nTotal: {len(tools)} tools from 2 Lambda-backed targets")
print("\nNote: Both Alice and Bob see the same tools.")
print("Access control is enforced at invocation time based on JWT group claims.")

## Cell 3: Alice Asks for Weather (ALLOWED)

Alice is in the `analysts` group. The Cedar policy allows **all authenticated users** to access `WeatherTools`. This should succeed.

In [ ]:
print("=" * 60)
print("ALICE (analyst) asks: 'What's the weather in Sydney?'")
print("=" * 60)

response = alice_agent("What's the weather in Sydney?")

print("\n" + "=" * 60)
print("RESULT: Alice (analyst) CAN access weather data")
print("Cedar policy: WeatherTools → ALLOW all authenticated users")
print("=" * 60)

## Cell 4: Alice Asks for Finance Data (BLOCKED)

Alice asks for confidential revenue data. She's in the `analysts` group, **not** `finance-admins`. The agent checks her groups from the JWT and **refuses the request**.

**This is the key "whoa" moment — the user's Okta identity flows through the Gateway and determines what they can access.**

In [ ]:
print("=" * 60)
print("ALICE (analyst) asks: 'Show me the quarterly revenue data'")
print("=" * 60)

response = alice_agent("Show me the quarterly revenue data")

print("\n" + "=" * 60)
print("RESULT: Alice (analyst) BLOCKED from finance data!")
print(f"Alice's JWT groups: {alice_auth['claims'].get('groups', [])}")
print("Policy: FinanceTools requires 'finance-admins' group")
print("=" * 60)

## Cell 5: Bob Asks for Finance Data (ALLOWED)

Now Bob asks the **exact same question**. He's in the `finance-admins` group. The agent checks his JWT groups and **allows the request**.

**Same question, different user, different result — driven entirely by Okta identity.**

In [ ]:
print("=" * 60)
print("BOB (finance-admin) asks: 'Show me the quarterly revenue data'")
print("=" * 60)

response = bob_agent("Show me the quarterly revenue data")

print("\n" + "=" * 60)
print("RESULT: Bob (finance-admin) CAN access finance data!")
print(f"Bob's JWT groups: {bob_auth['claims'].get('groups', [])}")
print("Same question as Alice. Same Gateway. Same tools.")
print("The ONLY difference: Bob's JWT has 'finance-admins' group")
print("=" * 60)

## Cell 6: Architecture Summary

What we just demonstrated:

In [ ]:
print("""
==================================================================
          AgentCore Gateway Demo - Summary
==================================================================

  ARCHITECTURE:

    User (CLI)
      |
      | username + password (Resource Owner Password grant)
      v
    Okta Token Endpoint --> JWT (with groups, client_id claims)
      |
      v
    Strands Agent (JWT as Bearer token)
      |
      | MCP (StreamableHTTP + Bearer JWT)
      v
    AgentCore Gateway
      |-- Ingress Auth: JWT validation (signature, audience, client_id)
      |-- Cedar Policy Engine (LOG_ONLY)
      |-- Agent-level RBAC: groups claim in system prompt
      |
      +------------------+
      |                  |
      v                  v
    Lambda            Lambda
   (Weather)         (Finance)

  AUTH FLOW:
    1. User authenticates via Okta ROPC grant -> JWT with groups claim
    2. Strands Agent attaches JWT as Bearer token on MCP connection
    3. Gateway validates JWT via Okta JWKS endpoint
    4. Cedar policies evaluated in LOG_ONLY mode (logged, not enforced)
    5. Agent checks user's groups from JWT claims in system prompt
    6. ALLOW or DENY based on group membership

  WHAT WE DEMONSTRATED:
    Alice (analyst)      -> Weather data  -> ALLOWED
    Alice (analyst)      -> Finance data  -> BLOCKED (not in finance-admins)
    Bob (finance-admin)  -> Finance data  -> ALLOWED

  KEY VALUE:
    - ONE Gateway manages multiple Lambda-backed MCP tools
    - Direct Okta OIDC -- no Cognito intermediary
    - User identity propagates through the entire chain
    - Group-based access control via JWT claims
    - Zero auth code in MCP servers

==================================================================
""")

# Cleanup MCPClients and httpx clients
alice_client.__exit__(None, None, None)
bob_client.__exit__(None, None, None)
print("Agents disconnected. Run cleanup in 01_setup.ipynb when done.")